# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets with their '@id', name, and fields
record_set_info = []
for rs in dataset.metadata.record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', f"RecordSet {rs_id}")
    fields = rs.get('fields', [])
    if isinstance(fields, dict):  # not typical, but check for singleton
        fields = [fields]
    field_ids = [fld['@id'] for fld in fields]
    record_set_info.append({'@id': rs_id, 'name': rs_name, 'fields': field_ids})
    print(f"RecordSet '@id': {rs_id}\n  name: {rs_name}\n  @id of fields: {field_ids}\n")

# For deeper exploration, get fields/columns' labels and types
print('--- Listing all fields and columns for each record set ---')
for rs in dataset.metadata.record_sets:
    rs_id = rs['@id']
    print(f"\nRecordSet '@id': {rs_id}")
    fields = rs.get('fields', [])
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        fld_id = fld['@id']
        fld_name = fld.get('name', fld_id)
        dtype = fld.get('dataType', 'Unknown type')
        columns = fld.get('columns', [])
        col_ids = [col['@id'] for col in columns] if isinstance(columns, list) else ([columns['@id']] if columns else [])
        print(f"  Field '@id': {fld_id} (name: {fld_name}) | type: {dtype}\n    columns: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: Load each record set into a DataFrame, keeping track of field IDs
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# For demonstration, consider the first record set (adjust as appropriate):
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Columns in DataFrame for RecordSet '@id': {main_record_set_id}\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations.

In [ ]:
# Explore columns and select a numeric field
import numpy as np
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print('Available columns:', df.columns.tolist())
    # Try auto-selecting a numeric field based on dtype or by common names
    numeric_candidates = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
    if not numeric_candidates:
        # Try to coerce columns possibly containing numeric data
        trial_cols = ['MW', 'coverage_percent', 'peptide_count', 'abundance', 'Normalized_Abundance']
        for col in trial_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                numeric_candidates.append(col)
    print('Numeric candidates:', numeric_candidates)
    numeric_field = numeric_candidates[0] if numeric_candidates else None

    if numeric_field is not None:
        # Filter for values above a threshold
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:\n", filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try groupby a category field if available
        group_candidates = [c for c in df.columns if df[c].dtype=='O' or df[c].dtype.name.startswith('category')]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):\n", grouped_df.head())
    else:
        print('No numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field is not None:
    df_plot = dataframes[main_record_set_id]

    plt.figure(figsize=(8, 5))
    sns.histplot(df_plot[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If a group_field was found, also show a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df_plot, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We reviewed the record sets and their fields using `@id`, loaded data into DataFrames, applied simple filtering and normalization to a numeric measurement, and visualized the distribution of a key field. This workflow can be extended with domain-specific analyses, leveraging the clear field and record semantics provided by Croissant schemas.